# Statistical Testing

`experiment.ipynb`에서 저장한 `results/experiment_results.csv`를 로드해  
**Paired t-test**로 모델 간 성과 차이의 유의성을 검정한다.

| 검정 | 비교 | 귀무가설 |
|---|---|---|
| RQ1 | `Graph-L1` vs `SampleGMV` | E2E 파이프라인의 효과 없음 |
| RQ2 | `Graph-L1` vs `Graph-L2`  | 손실함수 종류의 효과 없음 |
| RQ3 | `Graph-L1` vs `MLP-L1`   | 그래프 구조의 효과 없음 |

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── 데이터 로드 ───────────────────────────────────────────────────────────
df = pd.read_csv('results/experiment_results.csv')

STRATEGY_ORDER = ['EW', 'SampleGMV', 'MLP-L1', 'Graph-L1', 'Graph-L2']
COLORS = {
    'EW':         'gray',
    'SampleGMV':  'steelblue',
    'MLP-L1':     'mediumpurple',
    'Graph-L1':   'tomato',
    'Graph-L2':   'seagreen',
}
strategies = [s for s in STRATEGY_ORDER if s in df['strategy'].unique()]
METRICS    = ['Ann. Return', 'Ann. Vol', 'Sharpe', 'Max DD', 'Calmar']
N_RUNS     = df['seed'].nunique()

def get_values(s: str, k: str) -> np.ndarray:
    """seed 오름차순으로 정렬된 metric 배열 반환"""
    return df[df['strategy'] == s].sort_values('seed')[k].values

print(f'Loaded : {len(df)} rows')
print(f'Seeds  : {N_RUNS}  ({df["seed"].min()} ~ {df["seed"].max()})')
print(f'Strategies: {strategies}')
df.head(len(strategies))

## 1. Descriptive Statistics

In [ ]:
# ── Mean ± Std 요약 테이블 ────────────────────────────────────────────────
rows = []
for s in strategies:
    row = {'Strategy': s}
    for k in METRICS:
        v = get_values(s, k)
        row[k] = f'{v.mean():+.4f} ± {v.std():.4f}'
    rows.append(row)

desc = pd.DataFrame(rows).set_index('Strategy')
print(f'Mean ± Std  (N = {N_RUNS} seeds)')
print('=' * (14 + 20 * len(METRICS)))
print(desc.to_string())

## 2. Paired t-test

In [ ]:
# ── Paired t-test ─────────────────────────────────────────────────────────
# 각 seed에서 두 모델은 동일한 시장 데이터를 사용 → paired
# H0: mean(A - B) = 0   (two-sided)

PAIRS = [
    ('Graph-L1', 'SampleGMV', 'RQ1  E2E   vs Sample'),
    ('Graph-L1', 'Graph-L2',  'RQ2  L1    vs L2    '),
    ('Graph-L1', 'MLP-L1',    'RQ3  GNN   vs MLP   '),
]

def sig_stars(p: float) -> str:
    if p < 0.001: return '***'
    if p < 0.01:  return '** '
    if p < 0.05:  return '*  '
    return '   '

# 결과를 DataFrame으로도 보관
ttest_rows = []

print(f'Paired t-test  (N={N_RUNS} seeds,  two-sided)')
print(f'Significance: * p<0.05   ** p<0.01   *** p<0.001')

for s1, s2, label in PAIRS:
    print(f'\n{"─"*80}')
    print(f'{label}')
    for s in [s1, s2]:
        row_str = f'  {s:<14}'
        for k in METRICS:
            v = get_values(s, k)
            row_str += f'  {v.mean():>+7.4f}±{v.std():.4f}'
        print(row_str)
    row_str = f'  {"t-stat / p":<14}'
    for k in METRICS:
        x1, x2 = get_values(s1, k), get_values(s2, k)
        t, p   = stats.ttest_rel(x1, x2)
        row_str += f'  {t:>+7.3f}{sig_stars(p)}{p:.3f}'
        ttest_rows.append({'RQ': label.split()[0], 'Model A': s1, 'Model B': s2,
                           'Metric': k, 't-stat': round(t, 4), 'p-value': round(p, 4),
                           'sig': sig_stars(p).strip()})
    print(row_str)

print(f'\n{"─"*80}')
print('t > 0 → 왼쪽 모델이 더 높음   t < 0 → 오른쪽 모델이 더 높음')

df_ttest = pd.DataFrame(ttest_rows)
df_ttest.to_csv('results/ttest_results.csv', index=False)
print('\nSaved → results/ttest_results.csv')

## 3. Distribution Visualization

In [ ]:
# ── Violin plot: metric별 분포 ────────────────────────────────────────────
fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 5))

for ax, k in zip(axes, METRICS):
    data   = [get_values(s, k) for s in strategies]
    parts  = ax.violinplot(data, positions=range(len(strategies)),
                           showmedians=True, showextrema=True)
    for pc, s in zip(parts['bodies'], strategies):
        pc.set_facecolor(COLORS.get(s, 'blue'))
        pc.set_alpha(0.65)
    for part_name in ('cmedians', 'cmins', 'cmaxes', 'cbars'):
        parts[part_name].set_linewidth(1.2)

    # scatter overlay
    for i, s in enumerate(strategies):
        v = get_values(s, k)
        ax.scatter([i] * len(v), v, color=COLORS.get(s, 'blue'),
                   s=25, zorder=3, alpha=0.85, edgecolors='white', linewidths=0.5)

    ax.set_xticks(range(len(strategies)))
    ax.set_xticklabels(strategies, rotation=30, ha='right', fontsize=9)
    ax.set_title(k, fontsize=11)
    ax.axhline(0, color='black', linewidth=0.7, linestyle='--', alpha=0.4)
    ax.grid(alpha=0.3, axis='y')

plt.suptitle(f'Metric Distribution across {N_RUNS} Seeds', fontsize=13)
plt.tight_layout()
plt.savefig('results/stat_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── p-value heatmap: RQ × Metric ─────────────────────────────────────────
pivot = df_ttest.pivot_table(index=['RQ'], columns='Metric', values='p-value')
pivot = pivot.reindex(columns=METRICS)

fig, ax = plt.subplots(figsize=(len(METRICS) * 1.8, len(PAIRS) * 1.2 + 0.8))
im = ax.imshow(pivot.values.astype(float), vmin=0, vmax=0.1,
               cmap='RdYlGn_r', aspect='auto')

for i in range(len(pivot)):
    for j in range(len(METRICS)):
        val = pivot.values[i, j]
        stars = sig_stars(val).strip()
        txt   = f'{val:.3f}\n{stars}' if stars else f'{val:.3f}'
        ax.text(j, i, txt, ha='center', va='center', fontsize=10,
                color='white' if val < 0.03 else 'black')

ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels(METRICS, fontsize=10)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_title('p-value Heatmap  (green = significant,  threshold = 0.05)', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='p-value')
plt.tight_layout()
plt.savefig('results/pvalue_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()